# Feature Engineering

Transform cleaned game-level data into per-side player features for model training.

Each game produces two rows — one for white, one for black — with that player's
eval-based features, board state at move 15, and a per-side blunder label.


In [1]:
import sys
import numpy as np
import pandas as pd
import chess
from pathlib import Path

sys.path.append(str(Path().resolve().parent))
from config import CLEANED_PATH, FEATURES_PATH, SNAPSHOT_MOVE, BLUNDER_THRESHOLD, BLUNDER_WINDOW, CAP

In [2]:
df = pd.read_parquet(CLEANED_PATH)
print(df.shape)
df.head(2)

(14541, 9)


,game_id,white_elo,black_elo,time_control,opening,result,moves,evals,blunder_label
0,https://lichess.org/oXyWZjvn,1802,1791,600,Sicilian Defense,1-0,"[e2e4, c7c5, g1f3, d7d6, b1c3, g8f6, f1c4, g7g...","[-32.0, 29.0, -27.0, 30.0, -29.0, 29.0, -19.0,...",1
1,https://lichess.org/clKpvLl6,1270,1337,600,Scandinavian Defense,1-0,"[e2e4, d7d5, b1c3, d5d4, c3b5, c7c6, b5a3, g8f...","[-31.0, 72.0, 25.0, -18.0, 69.0, -18.0, 25.0, ...",1


## Per-Side Blunder Labels

Split the game-level blunder into white_blunder and black_blunder.
White's evals are at even indices (0, 2, 4...), black's at odd indices (1, 3, 5...).
A blunder is a drop of >200cp between consecutive moves for that side within moves 15-25.


In [3]:
def side_has_blunder(evals, offset, start=SNAPSHOT_MOVE, end=SNAPSHOT_MOVE + BLUNDER_WINDOW, threshold=BLUNDER_THRESHOLD):
    window = [min(max(v, -CAP), CAP) for v in evals[start:end] if v == v]
    side = window[offset::2]
    for i in range(1, len(side)):
        if side[i-1] - side[i] > threshold:
            return 1
    return 0

df["white_blunder"] = df["evals"].apply(lambda e: side_has_blunder(e, offset=0))
df["black_blunder"] = df["evals"].apply(lambda e: side_has_blunder(e, offset=1))


In [4]:
print("White blunder rate:", df["white_blunder"].mean().round(4))
print("Black blunder rate:", df["black_blunder"].mean().round(4))

reconstructed = ((df["white_blunder"] == 1) | (df["black_blunder"] == 1)).astype(int)
mismatch = (reconstructed != df["blunder_label"]).sum()
print("Mismatches with original label:", mismatch)  # must be 0


White blunder rate: 0.1428
Black blunder rate: 0.1474
Mismatches with original label: 0


## Eval Features

Computed from Stockfish centipawn scores over moves 1-15 for each side.

- eval_at_snapshot: position score at move 15 from this player's perspective
- eval_volatility: std of evals in moves 1-15 (high = tactical/sharp game)
- eval_trend: average change per move over the last 5 of this side's evals (positive = improving)

Evals are capped at ±1000 to prevent mate score sentinels (±10000) from distorting features.
Black's eval is negated so it represents black's advantage, not white's.

In [12]:
def eval_features(evals, offset, color, snapshot=SNAPSHOT_MOVE):
    window = [min(max(v, -CAP), CAP) for v in evals[:snapshot] if v == v]
    side = window[offset::2]
    
    if len(side) == 0:
        return {"eval_at_snapshot": np.nan, "eval_volatility": np.nan, "eval_trend": np.nan}
    
    sign = 1 if color == chess.WHITE else -1
    
    recent = side[-5:]
    trend = (recent[-1] - recent[0]) / len(recent) if len(recent) > 1 else 0.0
    
    return {
        "eval_at_snapshot": sign * side[-1],
        "eval_volatility": float(np.std(side)),
        "eval_trend": sign * trend,
    }



## Board Features

Replay the first 15 moves using python-chess to reconstruct the exact board state.

- material_balance: this player's total piece value minus opponent's (Q=9, R=5, B=3, N=3, P=1)
- material_imbalance: absolute value of material_balance
- king_attackers: number of enemy pieces directly attacking the king square (proxy for king danger)
- has_castled: 1 if this player castled before move 15, detected via move history

Note: board replay is O(n) per game. With ~14k games x 2 sides this takes ~30-60s.


In [13]:
PIECE_VALUES = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
    chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0
}

def board_features(moves, color, snapshot=SNAPSHOT_MOVE):
    board = chess.Board()
    castling_moves = {"e1g1", "e1c1"} if color == chess.WHITE else {"e8g8", "e8c8"}
    has_castled = 0
    
    for move in moves[:snapshot]:
        if move in castling_moves:
            has_castled = 1
        board.push(chess.Move.from_uci(move))
    
    my_material = sum(PIECE_VALUES[p.piece_type] for p in board.piece_map().values() if p.color == color)
    opp_material = sum(PIECE_VALUES[p.piece_type] for p in board.piece_map().values() if p.color != color)
    material_balance = my_material - opp_material
    
    king_sq = board.king(color)
    king_attackers = len(board.attackers(not color, king_sq))
    
    return {
        "material_balance": material_balance,
        "material_imbalance": abs(material_balance),
        "king_attackers": king_attackers,
        "has_castled": has_castled,
    }


## Build Feature DataFrame

Expand each game into two rows (white and black) with all features and per-side blunder label.


In [14]:
records = []
for _, row in df.iterrows():
    for color, offset, elo_col, opp_col, blunder_col in [
        (chess.WHITE, 0, "white_elo", "black_elo", "white_blunder"),
        (chess.BLACK, 1, "black_elo", "white_elo", "black_blunder"),
    ]:
        ef = eval_features(row["evals"], offset, color)
        bf = board_features(row["moves"], color)
        records.append({
            "game_id": row["game_id"],
            "side": "white" if color == chess.WHITE else "black",
            "player_elo": row[elo_col],
            "elo_diff": row[elo_col] - row[opp_col],
            "time_control": row["time_control"],
            **ef,
            **bf,
            "blunder_label": row[blunder_col],
        })

features_df = pd.DataFrame(records)
print(features_df.shape)
features_df.head(4)

(29082, 13)


,game_id,side,player_elo,elo_diff,time_control,eval_at_snapshot,eval_volatility,eval_trend,material_balance,material_imbalance,king_attackers,has_castled,blunder_label
0,https://lichess.org/oXyWZjvn,white,1802,11,600,63.0,43.744821,16.4,0,0,0,0,1
1,https://lichess.org/oXyWZjvn,black,1791,-11,600,74.0,42.857619,20.6,0,0,0,0,0
2,https://lichess.org/clKpvLl6,white,1270,-67,600,306.0,106.602064,56.2,-1,1,0,0,0
3,https://lichess.org/clKpvLl6,black,1337,67,600,161.0,80.186263,28.6,1,1,0,0,1


## Feature Distributions & Sanity Checks


In [15]:
print(features_df.dtypes)
print("\nNull counts:\n", features_df.isnull().sum())
print("\nBlunder rate by side:")
print(features_df.groupby("side")["blunder_label"].mean().round(4))
print("\nBlunder rate by Elo bracket:")
features_df["elo_bracket"] = pd.cut(features_df["player_elo"], bins=[1000,1200,1400,1600,1800,2000,2500])
print(features_df.groupby("elo_bracket", observed=True)["blunder_label"].mean().round(4))

print("\nFeature summary:")
print(features_df.drop(columns=["game_id", "side", "elo_bracket", "blunder_label"]).describe().round(2))


game_id                   str
side                      str
player_elo              int64
elo_diff                int64
time_control            int64
eval_at_snapshot      float64
eval_volatility       float64
eval_trend            float64
material_balance        int64
material_imbalance      int64
king_attackers          int64
has_castled             int64
blunder_label           int64
dtype: object

Null counts:
 game_id               0
side                  0
player_elo            0
elo_diff              0
time_control          0
eval_at_snapshot      0
eval_volatility       0
eval_trend            0
material_balance      0
material_imbalance    0
king_attackers        0
has_castled           0
blunder_label         0
dtype: int64

Blunder rate by side:
side
black    0.1474
white    0.1428
Name: blunder_label, dtype: float64

Blunder rate by Elo bracket:
elo_bracket
(1000, 1200]    0.3071
(1200, 1400]    0.2446
(1400, 1600]    0.1664
(1600, 1800]    0.1116
(1800, 2000]    0.0673
(20

In [16]:
features_df.drop(columns=["elo_bracket"]).to_parquet(FEATURES_PATH, index=False)
print("Saved:", features_df.shape)


Saved: (29082, 14)
